In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
STORAGE_ACCOUNT = "adlsairbnbde"

In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsairbnbde.dfs.core.windows.net",
    "KEY HERE"
)
 
SILVER_LISTINGS_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/listings"
SILVER_CALENDAR_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/calendar"
BRONZE_VALIDATED_NEIGHBOURHOODS = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/neighbourhoods"
 
GOLD_BASE = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net"

In [0]:
silver_listings = spark.read.format("delta").load(SILVER_LISTINGS_PATH)
silver_calendar = spark.read.format("delta").load(SILVER_CALENDAR_PATH)
neighbourhoods_raw = spark.read.format("delta").load(BRONZE_VALIDATED_NEIGHBOURHOODS)
 
print(f"Silver listings rows: {silver_listings.count()}")
print(f"Silver calendar rows: {silver_calendar.count()}")
print(f"Neighbourhoods rows: {neighbourhoods_raw.count()}")

Silver listings rows: 85028
Silver calendar rows: 31089612
Neighbourhoods rows: 207


In [0]:

date_range = silver_calendar.agg(
    F.min("date").alias("min_date"), F.max("date").alias("max_date")
).collect()[0]
 
min_date = date_range["min_date"]
max_date = date_range["max_date"]
print(f"Calendar date range: {min_date} to {max_date}")
 
dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) as date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin([1, 7]))
    .withColumn("quarter", F.quarter("date"))
)
 
dim_date.write.format("delta").mode("overwrite").save(f"{GOLD_BASE}/dim_date")
print(f"dim_date rows: {dim_date.count()}")
display(dim_date.limit(5))

Calendar date range: 2025-09-14 to 2027-07-02
dim_date rows: 657


date,date_key,year,month,day,day_of_week,is_weekend,quarter
2025-09-14,20250914,2025,9,14,Sunday,true,3
2025-09-15,20250915,2025,9,15,Monday,false,3
2025-09-16,20250916,2025,9,16,Tuesday,false,3
2025-09-17,20250917,2025,9,17,Wednesday,false,3
2025-09-18,20250918,2025,9,18,Thursday,false,3


In [0]:
dim_neighbourhood = (
    neighbourhoods_raw
    .select(
        F.col("neighbourhood").alias("neighbourhood_name"),
        F.col("neighbourhood_group").alias("neighbourhood_group"),
        F.col("_source_city").alias("city"),
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("neighbourhood_name", "city").orderBy(F.desc("neighbourhood_group"))
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("neighbourhood_key", F.monotonically_increasing_id())
)

In [0]:

fact_listing_snapshot = (
    silver_listings
    .join(
        dim_neighbourhood,
        (silver_listings.neighbourhood == dim_neighbourhood.neighbourhood_name) &
        (silver_listings.city == dim_neighbourhood.city),
        "left"
    )
    .select(
        silver_listings.listing_id,
        silver_listings.host_id,
        F.col("neighbourhood_key"),
        silver_listings.room_type,
        silver_listings.accommodates,
        silver_listings.bedrooms,
        silver_listings.price,
        silver_listings.number_of_reviews,
        silver_listings.has_reviews,
        silver_listings.review_scores_rating,
        silver_listings.host_is_superhost,
        silver_listings.host_listings_count,
        silver_listings.city,
        silver_listings.quarter_label,
    )
)
 
fact_listing_snapshot.write.format("delta").mode("overwrite").partitionBy("city").save(
    f"{GOLD_BASE}/fact_listing_snapshot"
)
print(f"fact_listing_snapshot rows: {fact_listing_snapshot.count()}")
display(fact_listing_snapshot.groupBy("city", "quarter_label").count())

fact_listing_snapshot rows: 85028


city,quarter_label,count
lisbon,2025-Q3,25449
lisbon,2026-Q2,24876
barcelona,2025-Q3,19410
barcelona,2026-Q2,15293


In [0]:

fact_calendar_availability = (
    silver_calendar
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .select(
        F.col("listing_id"),
        F.col("date_key"),
        F.col("available"),
        F.col("has_valid_availability"),
        F.col("minimum_nights"),
        F.col("maximum_nights"),
        F.col("has_nights_anomaly"),
        F.col("city"),
        F.col("quarter_label"),
    )
)
 
fact_calendar_availability.write.format("delta").mode("overwrite").partitionBy(
    "city", "quarter_label"
).save(f"{GOLD_BASE}/fact_calendar_availability")
 
print(f"fact_calendar_availability rows: {fact_calendar_availability.count()}")


fact_calendar_availability rows: 31089612


In [0]:

latest_quarter_window = Window.partitionBy("city", "listing_id").orderBy(F.desc("quarter_label"))
 
latest_per_city = (
    silver_listings
    .withColumn("rn", F.row_number().over(latest_quarter_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
 
dim_listing_current = latest_per_city.select(
    "listing_id", "host_id", "neighbourhood", "room_type",
    "accommodates", "bedrooms", "city", "quarter_label"
).withColumn("is_current", F.lit(True))
 
dim_host_current = (
    latest_per_city
    .select("host_id", "host_since_date", "host_is_superhost", "host_listings_count", "city", "quarter_label")
    .dropDuplicates(["host_id", "city"])
    .withColumn("is_current", F.lit(True))
)
 
dim_listing_current.write.format("delta").mode("overwrite").save(f"{GOLD_BASE}/dim_listing_current")
dim_host_current.write.format("delta").mode("overwrite").save(f"{GOLD_BASE}/dim_host_current")
 
print(f"dim_listing_current rows: {dim_listing_current.count()}")
print(f"dim_host_current rows: {dim_host_current.count()}")
 

dim_listing_current rows: 52174
dim_host_current rows: 18208


In [0]:

print("Gold star schema build complete:")
print(f"  dim_date:                    {dim_date.count()} rows")
print(f"  dim_neighbourhood:           {dim_neighbourhood.count()} rows")
print(f"  fact_listing_snapshot:       {fact_listing_snapshot.count()} rows")
print(f"  fact_calendar_availability:  {fact_calendar_availability.count()} rows")
print(f"  dim_listing_current:        {dim_listing_current.count()} rows")
print(f"  dim_host_current:           {dim_host_current.count()} rows")

Gold star schema build complete:
  dim_date:                    657 rows
  dim_neighbourhood:           206 rows
  fact_listing_snapshot:       85028 rows
  fact_calendar_availability:  31089612 rows
  dim_listing_current:        52174 rows
  dim_host_current:           18208 rows
